# SAVA-CLIP journal extension experiments
Tuned Tip-Adapter-F, CoOp, ViT-L/14 backbone, zero-shot ensembles, sensitivity analysis, McNemar tests. Run with GPU T4 x2 and Internet on.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/code/extend.py', recursive=True)
assert hits, 'Add the sava-clip-code-v2 dataset with Add Input'
src = os.path.dirname(os.path.dirname(hits[0])); dst = '/kaggle/working/SAVA-CLIP'
if not os.path.exists(dst): shutil.copytree(src, dst)
os.chdir(dst + '/code'); print(os.listdir('.'))

In [ ]:
!pip install -q 'transformers>=4.45,<5' accelerate scikit-learn scipy
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

In [ ]:
%cd /kaggle/working
!test -d PlantVillage-Dataset || (git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset && cd PlantVillage-Dataset && git sparse-checkout set raw/color)
!test -d PlantDoc-Dataset || git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset
!echo PlantVillage $(find PlantVillage-Dataset/raw/color -type f | wc -l) PlantDoc $(find PlantDoc-Dataset/train PlantDoc-Dataset/test -type f | wc -l)

In [ ]:
%cd /kaggle/working/SAVA-CLIP/code
!python extend.py --pv_root /kaggle/working/PlantVillage-Dataset/raw/color --pd_root /kaggle/working/PlantDoc-Dataset --out results/ext 2>&1 | grep -v -e Warning -e rationality -e warnings.warn -e check_classification | tee log_extend.txt

In [ ]:
import zipfile, glob, os
os.chdir('/kaggle/working/SAVA-CLIP/code')
files = glob.glob('results/ext/results_ext.json') + glob.glob('results/ext/*/preds.npz') + ['log_extend.txt']
with zipfile.ZipFile('/kaggle/working/ext_outputs.zip', 'w') as z:
    for f in files:
        if os.path.exists(f): z.write(f)
print(files)